In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Load the dataset
file_path = 'RELIANCE.NS_Stock.xlsx'
df = pd.read_excel(file_path, sheet_name='Master Sheet', engine='openpyxl')

In [ ]:
columns_to_drop = ["Ticker", "Symbol", "Start Date", "End Date", "Open", "Close", "Change", "Search Link", "Prompt", "Response", "Stock Return"]
df = df.drop(columns=columns_to_drop)
df.head()

In [ ]:
# Feature Engineering
# Encode Keywords columns (using LabelEncoder for simplicity; can use more complex methods)
label_encoder = LabelEncoder()
df['Keywords'] = label_encoder.fit_transform(df['Keywords'].astype(str))
df['Positive Keywords'] = label_encoder.fit_transform(df['Positive Keywords'].astype(str))
df['Negative Keywords'] = label_encoder.fit_transform(df['Negative Keywords'].astype(str))

In [ ]:
# Process 'Summary' column using TF-IDF
tfidf = TfidfVectorizer(max_features=100)  # Adjust max_features as needed
summary_tfidf = tfidf.fit_transform(df['Summary'].astype(str)).toarray()

In [ ]:
# Combine all features
X = pd.concat([
    df[['Keywords', 'Positive Keywords', 'Negative Keywords', 'Sentiment']],
    pd.DataFrame(summary_tfidf)
], axis=1)

In [ ]:
# Convert all column names to strings
X.columns = X.columns.astype(str)

In [ ]:
# Target Variable
y = df['Bins'].apply(lambda x: 'U' if 'U' in x else 'D')  # Convert to binary 'U'/'D'

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Normalize numerical features
scaler = StandardScaler()

In [ ]:
# Convert X_train and X_test to DataFrame after scaling, keeping the column names
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
# Train Model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Predict
y_pred = model.predict(X_test)

In [ ]:
# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))